In [11]:
import requests
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import time

def fetch_pendle_historical_apys(market_dict, price_data_dict, output_dir='./',
                                 start_date=None, end_date=None, time_frame='hour'):
    """
    Fetch all historical APY data for given Pendle markets, merge with closest price,
    and save each as CSV. Automatically paginates using timestamp_start/end.

    Parameters:
    market_dict: dict mapping symbol -> {'pendle_address': market_id, 'asset_address': asset_addr}
    price_data_dict: dict mapping asset_address -> {'historical_price': list of [timestamp, price]}
    output_dir: directory to save CSV files
    start_date: optional datetime string (e.g., '2024-01-01') or timestamp; if None, fetches as far back as possible
    end_date: optional datetime string or timestamp; if None, uses current time
    time_frame: 'hour', 'day', or 'week' – determines chunk size (max 1440 points)
    """
    base_url = "https://api-v2.pendle.finance/core/v2/1/markets/{}/historical-data"
    
    # Convert start/end dates to timestamps
    if end_date is None:
        end_ts = int(time.time())
    else:
        end_ts = int(pd.Timestamp(end_date).timestamp())
    
    if start_date is None:
        start_ts = 0  # fetch as far back as possible
    else:
        start_ts = int(pd.Timestamp(start_date).timestamp())
    
    # Chunk size in seconds based on time_frame and max points (1440)
    if time_frame == 'hour':
        chunk_seconds = 60 * 24 * 3600  # 60 days
    elif time_frame == 'day':
        chunk_seconds = 1440 * 24 * 3600  # ~1440 days (about 4 years)
    elif time_frame == 'week':
        chunk_seconds = 1440 * 7 * 24 * 3600  # ~1440 weeks
    else:
        raise ValueError("time_frame must be 'hour', 'day', or 'week'")
    
    # Preprocess price data for fast lookup
    price_dfs = {}
    for addr, data in price_data_dict.items():
        if data.get('historical_price'):
            df = pd.DataFrame(data['historical_price'], columns=['timestamp', 'collateral_price'])
            df = df.sort_values('timestamp')
            price_dfs[addr] = df
    
    for symbol, info in market_dict.items():
        market_id = info['pendle_address']
        asset_addr = info['asset_address']
        
        all_results = []
        current_end = end_ts
        while current_end > start_ts:
            current_start = max(start_ts, current_end - chunk_seconds)
            
            # Format timestamps as ISO strings
            start_str = datetime.utcfromtimestamp(current_start).isoformat() + 'Z'
            end_str = datetime.utcfromtimestamp(current_end).isoformat() + 'Z'
            
            params = {
                'time_frame': time_frame,
                'timestamp_start': start_str,
                'timestamp_end': end_str,
                'fields': 'timestamp,maxApy,baseApy,underlyingApy,impliedApy,tvl'  # adjust fields as needed
            }
            
            url = base_url.format(market_id)
            try:
                response = requests.get(url, params=params)
                response.raise_for_status()
                data = response.json()
            except Exception as e:
                print(f"Failed to fetch data for {symbol} ({market_id}): {e}")
                break
            
            if not data.get("results"):
                print(f"No more results for {symbol} before {current_start}")
                break
            
            df_chunk = pd.DataFrame(data["results"])
            all_results.append(df_chunk)
            
            # Stop if we got less than 1440 points (means we've reached the beginning)
            if len(df_chunk) < 1440:
                break
            
            # Move the end backward for the next chunk
            # Use the earliest timestamp in this chunk minus 1 second to avoid overlap
            earliest_in_chunk = pd.to_datetime(df_chunk['timestamp']).min()
            current_end = int(earliest_in_chunk.timestamp()) - 1
        
        if not all_results:
            print(f"No data for {symbol} ({market_id})")
            continue
        
        # Combine all chunks
        df = pd.concat(all_results, ignore_index=True)
        df = df.drop_duplicates(subset=['timestamp']).sort_values('timestamp')
        
        # Rename columns
        df = df.rename(columns={
            "timestamp": "datetime_str",
            "baseApy": "base_apy",
            "impliedApy": "implied_apy",
            "underlyingApy": "underlying_apy",
            "tvl": "tvl"
        })
        
        df['datetime'] = pd.to_datetime(df['datetime_str'])
        df['timestamp'] = df['datetime'].astype(int) // 10**9
        df['symbol'] = symbol
        df['market_id'] = market_id
        df['asset_address'] = asset_addr
        
        # Merge closest price
        if asset_addr in price_dfs:
            price_df = price_dfs[asset_addr]
            df = df.sort_values('timestamp')
            price_df = price_df.sort_values('timestamp')
            df = pd.merge_asof(df, price_df, on='timestamp', direction='nearest')
        else:
            df['collateral_price'] = np.nan
        
        final_cols = ['timestamp', 'datetime', 'base_apy', 'implied_apy', 'underlying_apy', 'tvl',
                      'symbol', 'market_id', 'asset_address', 'collateral_price']
        df = df[final_cols]
        
        filename = f"{output_dir}/{symbol}.csv"
        df.to_csv(filename, index=False)
        print(f"Saved {len(df)} rows for {symbol} to {filename}")

In [12]:

import json

# Read the entire file
with open('/Users/yegortrussov/Documents/ml/lending_protocols/dataset_collection/data/common/assets_meta.json', 'r') as f:
    assets_meta = json.load(f)
market_dict = {
    "PT-siUSD-26MAR2026": {"pendle_address": "0x564f279b0226f60a40f1e4b8c596feb87c383bfa", "asset_address": "0xaF76B3AF3477E4a2cD0B7F80c3152108c19a25e5"},
    "PT-reUSD-25JUN2026": {"pendle_address": "0xf5929a1c332ceab7918a4593a43db2b9ac20095f", "asset_address": "0x3EAA0F0f0A5d3D595ae4e4b0D27f439d01c3E7b2"}
}
fetch_pendle_historical_apys(
    market_dict,
    assets_meta,
    "/Users/yegortrussov/Documents/ml/lending_protocols/dataset_collection/data/pt_yields"
)


Saved 1526 rows for PT-siUSD-26MAR2026 to /Users/yegortrussov/Documents/ml/lending_protocols/dataset_collection/data/pt_yields/PT-siUSD-26MAR2026.csv
Saved 2176 rows for PT-reUSD-25JUN2026 to /Users/yegortrussov/Documents/ml/lending_protocols/dataset_collection/data/pt_yields/PT-reUSD-25JUN2026.csv
